# HEAL-Summ-Lite: Lightweight Ethical Health Summarization Pipeline

## Architecture (4 Stages, Two-Shot Prompting)
```
Original Text (600-700 words, FULL — no pre-extraction)
  → [Stage 1] Two-Shot Phi-3 Mini Summarization  (1 LLM call, ~2-5 min)
  → [Stage 2] Readability Scoring (FKGL + FRE)    (Python, instant)
  → [Stage 3] 10 Rule-Based Heuristic Checks      (Python, instant)
  → [Stage 4] Tiered Human Review Decision         (Python, instant)
  → Results Tables + Severity Report
```

## Key Design Decisions
- **Two-shot prompting** — two examples teach the model to cover ALL sections (beginning, middle, end)
- **Full text input** — no pre-extraction (TextRank/TF-IDF removed entities and lowered readability)
- **Sentence-count prompt** — "write 8-10 sentences" instead of word count (small LLMs can count sentences, not words)
- **`num_predict=240`** tokens ≈ 180 word ceiling — model wraps up naturally (NOT trimming)
- **No trimming** — raw model output preserved. Heuristics flag deviations honestly
- **10 deterministic heuristics** — instant, reproducible, auditable quality checks


In [1]:
!pip install requests textstat tabulate --quiet
print("✓ Dependencies installed.")

✓ Dependencies installed.


## Step 2: Imports & Configuration

In [2]:
import json
import re
import time
import difflib
import requests
import textstat
from tabulate import tabulate
from health_texts import HEALTH_TEXTS

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "phi3:mini"
SUMMARIZATION_TEMPERATURE = 0.3
MAX_FKGL_GRADE = 14
MIN_WORD_COUNT = 120
MAX_WORD_COUNT = 180
FRE_MINIMUM = 30.0

print(f"  Model: {MODEL_NAME}")
print(f"  Total health-related texts available: {len(HEALTH_TEXTS)}")

  Model: phi3:mini
  Total health-related texts available: 5


## Step 3: Two-Shot Prompt & Ollama Interface

**Why two-shot?** Zero-shot: model has no example → paraphrases sequentially from paragraph 1 and never reaches later paragraphs. Two-shot: model sees TWO examples of good summaries that cover ALL sections → mimics this full-coverage pattern through **in-context learning**.

**Why not TextRank/TF-IDF pre-extraction?** Tested and rejected — removing sentences before the LLM sees them caused missing entities, lower readability, and insufficient word count. The LLM must see the FULL text to preserve all statistics, drug names, and safety caveats.

**Word count control (NO trimming):** Sentence-count prompt (8-10 sentences × 15-20 words) + `num_predict=240` tokens ≈ 180 word natural ceiling.


In [3]:
# =============================================================================
# TWO-SHOT EXAMPLES (~160 words each)
# =============================================================================
# Why this length matters:
#   The model mimics the LENGTH of examples it sees. If examples are 110 words,
#   the model generates ~110 words and stops (EOS token). If examples are ~160
#   words, the model targets ~160 words — landing in our 150-180 range.
#   This is called "in-context length learning" — the most reliable way to
#   control output length in small LLMs without trimming.
#
# Why num_predict=300?
#   Safety net ONLY. The examples control actual stopping (~160 words).
#   num_predict=300 (~225 words) is set ABOVE the target so the model
#   has room to finish its last sentence and generate EOS naturally.
#   If it ever hits 300, clean_incomplete_output() removes the fragment.
# =============================================================================

EXAMPLE_1_INPUT = """Asthma is a chronic respiratory condition that affects the airways in the lungs. During an asthma attack, the airways become inflamed and narrowed, making it difficult to breathe. Common triggers include allergens such as pollen and dust mites, cold air, physical exercise, and respiratory infections. According to WHO, approximately 262 million people worldwide were affected by asthma in 2019, and it caused 455,000 deaths that year. Asthma is the most common chronic disease among children. While asthma cannot be cured, proper management with inhaled corticosteroids and bronchodilators can control symptoms effectively. Regular monitoring of symptoms and lung function helps prevent severe attacks. Patients should avoid known triggers, follow their asthma action plan, get a flu vaccine each year, and consult a healthcare professional for personalized treatment."""

EXAMPLE_1_OUTPUT = """Asthma is a chronic respiratory condition where the airways become inflamed and narrowed during attacks, causing significant breathing difficulty. Common triggers include allergens such as pollen and dust mites, cold air, physical exercise, and respiratory infections. The WHO reports that approximately 262 million people worldwide were affected by asthma in 2019, and it caused 455,000 deaths globally that year. Asthma is the most common chronic disease among children, requiring careful and consistent long-term management. While no cure currently exists, inhaled corticosteroids and bronchodilators can effectively control asthma symptoms when used as prescribed. Patients should identify and avoid their known triggers and follow their personalized asthma action plan consistently for best results. Regular monitoring of symptoms and lung function helps prevent severe asthma attacks and reduces emergency hospital visits significantly. Getting a flu vaccine each year is also recommended because respiratory infections can trigger dangerous asthma episodes. Consult a healthcare professional before making medical decisions about asthma treatment or changing any medication plans."""

EXAMPLE_2_INPUT = """Tuberculosis (TB) is caused by bacteria called Mycobacterium tuberculosis that most often affect the lungs. TB is spread through the air when infected people cough, sneeze, or spit. About a quarter of the global population is estimated to have been infected with TB bacteria. The WHO reported 10.6 million new TB cases and 1.3 million deaths from TB in 2022. TB is preventable and curable. About 85% of people who develop TB disease can be successfully treated with a 6-month drug regimen. TB treatment has averted over 75 million deaths since the year 2000. Multidrug-resistant TB remains a public health crisis, with only about 2 in 5 affected people accessing treatment. The BCG vaccine is given to children in many countries to prevent severe forms of childhood TB. Early detection through diagnostic testing is essential because delayed treatment increases transmission risk. Consult a healthcare professional if you experience persistent cough, fever, or unexplained weight loss."""

EXAMPLE_2_OUTPUT = """Tuberculosis is a serious bacterial infection caused by Mycobacterium tuberculosis that primarily affects the lungs and spreads through airborne droplets. About one quarter of the entire global population is estimated to have been infected with TB bacteria at some point. The WHO reported approximately 10.6 million new TB cases and 1.3 million TB-related deaths worldwide in the year 2022. TB is both preventable and curable, with about 85 percent of patients successfully treated using a standard 6-month drug regimen. Since the year 2000, TB treatment programs have averted over 75 million deaths globally, demonstrating remarkable progress in disease control. However, multidrug-resistant TB remains a serious public health crisis, with only about 2 in 5 affected people accessing treatment. The BCG vaccine is administered to children in many countries around the world to help prevent severe forms of childhood TB. Early detection through diagnostic testing is essential because delayed treatment increases the risk of transmission to others significantly. Consult a healthcare professional before making medical decisions about TB symptoms, testing, or treatment options."""


SUMMARIZATION_SYSTEM_PROMPT = """You write health information for people who did not finish high school.

CRITICAL RULES - FOLLOW EXACTLY:

1. SENTENCES: Write 8-10 sentences. Each sentence MUST be 8-12 words. 
   - NO sentence over 15 words. Break long sentences into two.
   
2. WORDS: Use only simple words a child would know.
   - NO: transmitted, intervention, disproportionately, chemoprevention
   - YES: spread, help, mostly, medicine
   
3. LISTS: Never put more than 3 items in a list.
   - BAD: "infants, children, pregnant women, HIV patients, migrants, travelers"
   - GOOD: "babies, young children, and pregnant women are at high risk. So are people with HIV and travelers."

4. NUMBERS: Keep all numbers exactly as written.

5. HARD WORDS: If you must use a medical term, explain it.
   - "Insecticide-treated nets (ITNs) are bed nets with bug spray on them."

6. END: If treatments are mentioned, say "Talk to a doctor before making health choices."

Write simply. Short sentences. Easy words."""


def build_summarization_prompt(original_text):
    """Builds a two-shot prompt — two ~160-word examples teach full-coverage summarization."""
    return f"""{SUMMARIZATION_SYSTEM_PROMPT}

Here are two examples of good health summaries:

EXAMPLE 1 INPUT:
{EXAMPLE_1_INPUT}

EXAMPLE 1 SUMMARY:
{EXAMPLE_1_OUTPUT}

EXAMPLE 2 INPUT:
{EXAMPLE_2_INPUT}

EXAMPLE 2 SUMMARY:
{EXAMPLE_2_OUTPUT}

Now summarize the following health text the same way — cover ALL topics from beginning to end:

INPUT TEXT:
{original_text}

SUMMARY:"""


def clean_incomplete_output(text):
    """
    Removes ONLY the trailing incomplete sentence fragment.
    This is NOT trimming — it removes broken text caused by num_predict cutoff.
    If the output is already complete (ends with . ! ?), this does nothing.
    """
    text = text.strip()
    if text and text[-1] in '.!?':
        return text  # Already complete — no change
    last_end = max(text.rfind('.'), text.rfind('!'), text.rfind('?'))
    if last_end > 0:
        return text[:last_end + 1]
    return text


def call_ollama(prompt, temperature):
    """Sends a prompt to Phi-3 Mini via Ollama local API."""
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 0.9,
            "num_predict": 300,     # Safety net — examples control actual length (~160 words)
            "num_ctx": 4096,
            "num_thread": 4,
            "repeat_penalty": 1.1,
        },
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=600)
        if response.status_code != 200:
            print(f"\n  [ERROR] Ollama returned {response.status_code}")
            print(f"  [ERROR] Response: {response.text[:300]}")
        response.raise_for_status()
    except requests.exceptions.ConnectionError:
        print("\n[ERROR] Cannot connect to Ollama. Make sure it is running.")
        return None
    except requests.exceptions.ReadTimeout:
        print("\n[ERROR] Timeout (>10 min). Close other apps to free RAM.")
        return None
    except requests.exceptions.HTTPError as e:
        print(f"\n[ERROR] Ollama error: {e}")
        return None
    try:
        result = response.json()
        return result.get("response", "").strip() or None
    except Exception as e:
        print(f"\n[ERROR] Invalid Ollama response: {e}")
        return None


# Quick connectivity test
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f"✓ Ollama is running. Models: {models}")
except:
    print("⚠ Cannot connect to Ollama. Run: ollama serve")



✓ Ollama is running. Models: ['phi3:mini']


## Step 4: Readability Scoring

In [4]:
def compute_readability(text):
    if not text or not text.strip():
        return {"fkgl": None, "fre": None}
    return {
        "fkgl": round(textstat.flesch_kincaid_grade(text), 1),
        "fre": round(textstat.flesch_reading_ease(text), 1),
    }

print("✓ Readability scoring ready.")

✓ Readability scoring ready.


## Step 5: 10 Rule-Based Heuristic Checks

| # | Check | Catches | Severity |
|---|---|---|---|
| 1 | Numeric Coverage | Missing statistics | WARNING |
| 2 | Entity Coverage | Missing drug/org names | WARNING |
| 3 | Compression Ratio | Barely shorter or too aggressive | WARNING |
| 4 | Caveat Presence | Treatment without safety disclaimer | CRITICAL |
| 5 | Negation Flip | "does NOT spread" → "spreads" | CRITICAL |
| 6 | Hedging Preservation | "may cause" → "causes" | WARNING |
| 7 | Source Attribution | Missing WHO/CDC/NHS | INFO |
| 8 | Sentence Count | Too few or too many | WARNING |
| 9 | Semantic Similarity | Hallucination or copy-paste | CRITICAL |
| 10 | Sensitive Topics | Missing crisis helpline | CRITICAL |

In [5]:
# --- HELPER: Extract numbers from text ---
def extract_numbers(text):
    """Extracts all numeric values including percentages, decimals, comma-separated."""
    raw = re.findall(r'\d[\d,]*\.?\d*%?', text)
    return set(num.replace(",", "") for num in raw)


# --- HELPER: Extract key entities ---
def extract_key_entities(text):
    """Extracts organization names, drug names, and disease terms."""
    entities = set()
    orgs = [
        "WHO", "CDC", "NHS", "World Health Organization",
        "Centers for Disease Control", "National Health Service",
        "American Psychological Association", "National Institute of Mental Health",
        "National Sleep Foundation",
    ]
    for org in orgs:
        if org.lower() in text.lower():
            entities.add(org)
    drugs = re.findall(
        r'\b(ACE inhibitors?|ARBs?|calcium channel blockers?|diuretics?|'
        r'beta blockers?|antiviral drugs?|RTS,S/AS01|Mosquirix|R21/Matrix-M|'
        r'insecticide-treated nets?|ITNs?|IRS|insulin|chemoprevention|'
        r'oseltamivir|Tamiflu|zanamivir|Relenza|peramivir|Rapivab|'
        r'baloxavir marboxil|Xofluza|artemisinin|ACT|'
        r'enalapril|lisinopril|amlodipine|CBT|'
        r'cognitive behavioral therapy)\b',
        text, re.IGNORECASE
    )
    for drug in drugs:
        entities.add(drug.lower())
    diseases = re.findall(
        r'\b(diabetes|malaria|influenza|flu|hypertension|'
        r'high blood pressure|pneumonia|sepsis|HIV|AIDS|'
        r'obesity|retinopathy|neuropathy|'
        r'anxiety|depression|stroke|heart attack)\b',
        text, re.IGNORECASE
    )
    for disease in diseases:
        entities.add(disease.lower())
    return entities

print("✓ Helper functions loaded.")

✓ Helper functions loaded.


In [6]:
# =============================================================================
# CHECK 1: Numeric Coverage
# =============================================================================
def check_numeric_coverage(original, summary):
    """What % of numbers from the original appear in the summary?"""
    orig_nums = extract_numbers(original)
    summ_nums = extract_numbers(summary)
    if not orig_nums:
        return {"status": "PASS", "severity": "INFO",
                "reason": "No numbers in original", "detail": {}}
    missing = orig_nums - summ_nums
    coverage = round((len(orig_nums) - len(missing)) / len(orig_nums) * 100, 1)
    if coverage < 30:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Only {coverage}% of numbers preserved ({len(missing)} missing)",
                "detail": {"coverage": coverage, "missing": list(missing)[:5]}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"{coverage}% of numbers preserved",
            "detail": {"coverage": coverage, "missing": list(missing)[:5]}}


# =============================================================================
# CHECK 2: Entity Coverage
# =============================================================================
def check_entity_coverage(original, summary):
    """What % of key entities from the original appear in the summary?"""
    orig_ents = extract_key_entities(original)
    summ_ents = extract_key_entities(summary)
    if not orig_ents:
        return {"status": "PASS", "severity": "INFO",
                "reason": "No key entities detected", "detail": {}}
    orig_lower = {e.lower() for e in orig_ents}
    summ_lower = {e.lower() for e in summ_ents}
    missing = orig_lower - summ_lower
    coverage = round((len(orig_lower) - len(missing)) / len(orig_lower) * 100, 1)
    if coverage < 40:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Only {coverage}% of entities preserved ({len(missing)} missing)",
                "detail": {"coverage": coverage, "missing": list(missing)[:5]}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"{coverage}% of entities preserved",
            "detail": {"coverage": coverage, "missing": list(missing)[:5]}}


# =============================================================================
# CHECK 3: Compression Ratio
# =============================================================================
def check_compression_ratio(original, summary):
    """Is the summary appropriately shorter than the original?"""
    orig_words = len(original.split())
    summ_words = len(summary.split())
    ratio = round(summ_words / orig_words, 2) if orig_words > 0 else 0
    if ratio > 0.85:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Ratio {ratio} — summary barely shorter than original",
                "detail": {"ratio": ratio}}
    if ratio < 0.10:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Ratio {ratio} — summary may be too aggressive",
                "detail": {"ratio": ratio}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"Compression ratio {ratio} — within acceptable range",
            "detail": {"ratio": ratio}}


# =============================================================================
# CHECK 4: Caveat / Safety Language Check
# =============================================================================
def check_caveat_presence(original, summary):
    """If original discusses treatments, does summary have safety language?"""
    treatment_keywords = [
        "treatment", "treated", "medication", "medicine", "prescribe",
        "drug", "dosage", "dose", "vaccine", "therapy", "antiviral",
        "inhibitor", "surgery", "insulin", "chemoprevention",
    ]
    caveat_phrases = [
        "consult", "healthcare professional", "medical professional",
        "doctor", "physician", "seek medical", "professional advice",
        "health provider", "medical advice", "medical decisions",
    ]
    has_treatment = any(kw in original.lower() for kw in treatment_keywords)
    has_caveat = any(phrase in summary.lower() for phrase in caveat_phrases)
    if has_treatment and not has_caveat:
        return {"status": "FLAG", "severity": "CRITICAL",
                "reason": "Treatment content found but NO safety caveat in summary",
                "detail": {"has_treatment": True, "has_caveat": False}}
    if has_treatment and has_caveat:
        return {"status": "PASS", "severity": "INFO",
                "reason": "Treatment content found; safety caveat present",
                "detail": {"has_treatment": True, "has_caveat": True}}
    return {"status": "PASS", "severity": "INFO",
            "reason": "No treatment content — caveat not required",
            "detail": {"has_treatment": False, "has_caveat": has_caveat}}


# =============================================================================
# CHECK 5: Negation Flip Detection
# =============================================================================
def check_negation_flip(original, summary):
    """Detects if critical negations from the original are missing in summary."""
    negation_patterns = [
        r"(not|no|never|neither|nor|cannot|can't|don't|doesn't|didn't|won't|shouldn't|isn't|aren't)\s+(\w+(?:\s+\w+){0,4})",
    ]
    original_negations = []
    for pattern in negation_patterns:
        matches = re.findall(pattern, original.lower())
        for match in matches:
            neg_word, context = match
            original_negations.append((neg_word, context.strip()))
    if not original_negations:
        return {"status": "PASS", "severity": "INFO",
                "reason": "No critical negations found in original", "detail": {}}
    potential_flips = []
    for neg_word, context in original_negations:
        if len(context.split()) < 2:
            continue
        if context in summary.lower():
            context_pos = summary.lower().find(context)
            window_start = max(0, context_pos - 30)
            window = summary.lower()[window_start:context_pos + len(context) + 10]
            neg_words = ["not", "no", "never", "cannot", "can't", "don't",
                         "doesn't", "didn't", "won't", "shouldn't", "isn't", "aren't"]
            if not any(nw in window for nw in neg_words):
                potential_flips.append(f"'{neg_word} {context}' may be flipped")
    if potential_flips:
        return {"status": "FLAG", "severity": "CRITICAL",
                "reason": f"Possible negation flip: {potential_flips[0]}",
                "detail": {"flips": potential_flips}}
    return {"status": "PASS", "severity": "INFO",
            "reason": "No negation flips detected",
            "detail": {"checked": len(original_negations)}}


# =============================================================================
# CHECK 6: Hedging Language Preservation
# =============================================================================
def check_hedging_preservation(original, summary):
    """Checks if uncertain/speculative language is preserved."""
    hedging_phrases = [
        "may", "might", "could", "possibly", "potentially",
        "suggests", "preliminary", "estimated", "approximately",
        "believed to", "thought to", "appears to", "seems to",
        "is associated with", "may contribute", "more research",
    ]
    orig_hedges = [h for h in hedging_phrases if h in original.lower()]
    if not orig_hedges:
        return {"status": "PASS", "severity": "INFO",
                "reason": "No hedging language in original", "detail": {}}
    missing_hedges = [h for h in orig_hedges if h not in summary.lower()]
    preservation = round((len(orig_hedges) - len(missing_hedges)) / len(orig_hedges) * 100, 1)
    if preservation < 30 and len(orig_hedges) >= 2:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Only {preservation}% hedging preserved — claims may sound more definitive",
                "detail": {"original_hedges": orig_hedges, "missing": missing_hedges}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"{preservation}% hedging language preserved",
            "detail": {"original_hedges": orig_hedges, "preserved": len(orig_hedges) - len(missing_hedges)}}


# =============================================================================
# CHECK 7: Source Attribution
# =============================================================================
def check_source_attribution(original, summary, source_name):
    """Does the summary mention the source organization?"""
    source_terms = []
    if "WHO" in source_name or "World Health" in source_name:
        source_terms = ["who", "world health organization"]
    elif "CDC" in source_name or "Centers for Disease" in source_name:
        source_terms = ["cdc", "centers for disease control"]
    elif "NHS" in source_name or "National Health Service" in source_name:
        source_terms = ["nhs", "national health service"]
    if not source_terms:
        return {"status": "PASS", "severity": "INFO",
                "reason": "Source type not tracked", "detail": {}}
    has_attribution = any(term in summary.lower() for term in source_terms)
    if not has_attribution:
        return {"status": "FLAG", "severity": "INFO",
                "reason": f"Summary does not mention source ({source_name})",
                "detail": {"source": source_name}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"Source attribution present ({source_name})",
            "detail": {"source": source_name}}


# =============================================================================
# CHECK 8: Sentence Count
# =============================================================================
def check_sentence_count(summary):
    """Verifies summary has a reasonable number of sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', summary.strip())
    count = len(sentences)
    if count < 3:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Only {count} sentences — summary may lack detail",
                "detail": {"sentence_count": count}}
    if count > 12:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"{count} sentences — summary may not be condensed enough",
                "detail": {"sentence_count": count}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"{count} sentences — appropriate for summary length",
            "detail": {"sentence_count": count}}


# =============================================================================
# CHECK 9: Semantic Similarity
# =============================================================================
def check_semantic_similarity(original, summary):
    """Computes text similarity between original and summary."""
    orig_clean = re.sub(r'\s+', ' ', original.lower().strip())
    summ_clean = re.sub(r'\s+', ' ', summary.lower().strip())
    similarity = round(difflib.SequenceMatcher(None, orig_clean, summ_clean).ratio() * 100, 1)
    if similarity < 15:
        return {"status": "FLAG", "severity": "CRITICAL",
                "reason": f"Similarity {similarity}% — possible hallucinated content",
                "detail": {"similarity_pct": similarity}}
    if similarity > 85:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Similarity {similarity}% — too close to copy-paste",
                "detail": {"similarity_pct": similarity}}
    return {"status": "PASS", "severity": "INFO",
            "reason": f"Similarity {similarity}% — good faithfulness/compression balance",
            "detail": {"similarity_pct": similarity}}


# =============================================================================
# CHECK 10: Sensitive Topic Flag
# =============================================================================
def check_sensitive_topics(original, summary):
    """Flags if sensitive health topics need special handling."""
    sensitive_keywords = [
        "suicide", "self-harm", "self harm", "crisis", "addiction",
        "overdose", "eating disorder", "substance abuse", "mental health crisis",
        "domestic violence", "abuse",
    ]
    resource_patterns = [
        r"988", r"911", r"helpline", r"hotline", r"crisis line",
        r"lifeline", r"emergency", r"call or text",
    ]
    has_sensitive = any(kw in original.lower() for kw in sensitive_keywords)
    if not has_sensitive:
        return {"status": "PASS", "severity": "INFO",
                "reason": "No sensitive topics detected", "detail": {}}
    orig_resources = [r for r in resource_patterns if re.search(r, original.lower())]
    summ_resources = [r for r in resource_patterns if re.search(r, summary.lower())]
    missing_resources = set(orig_resources) - set(summ_resources)
    if missing_resources:
        return {"status": "FLAG", "severity": "CRITICAL",
                "reason": "Sensitive topic detected but crisis resources missing from summary",
                "detail": {"missing_resources": list(missing_resources)}}
    return {"status": "PASS", "severity": "INFO",
            "reason": "Sensitive topic detected; crisis resources preserved",
            "detail": {"resources_found": summ_resources}}


# =============================================================================
# RUN ALL 10 HEURISTICS
# =============================================================================
def run_all_heuristics(original, summary, source_name):
    """Runs all 10 rule-based heuristic checks and returns results dict."""
    return {
        "numeric_coverage":     check_numeric_coverage(original, summary),
        "entity_coverage":      check_entity_coverage(original, summary),
        "compression_ratio":    check_compression_ratio(original, summary),
        "caveat_presence":      check_caveat_presence(original, summary),
        "negation_flip":        check_negation_flip(original, summary),
        "hedging_preservation": check_hedging_preservation(original, summary),
        "source_attribution":   check_source_attribution(original, summary, source_name),
        "sentence_count":       check_sentence_count(summary),
        "semantic_similarity":  check_semantic_similarity(original, summary),
        "sensitive_topics":     check_sensitive_topics(original, summary),
    }

print("✓ All 10 heuristic checks + run_all_heuristics loaded.")

✓ All 10 heuristic checks + run_all_heuristics loaded.


## Step 6: Tiered Human Review

| Tier | Condition | Action |
|---|---|---|
| **CRITICAL** | Any CRITICAL flag | Always escalate |
| **ESCALATE** | 2+ WARNING flags | Escalate |
| **REVIEW** | 1 WARNING flag | Soft flag |
| **APPROVED** | No flags | Safe to publish |

In [7]:
def decide_human_review(word_count, readability, heuristics):
    """Determines escalation level based on severity tiers."""
    critical_reasons = []
    warning_reasons = []
    info_notes = []

    for check_name, result in heuristics.items():
        if result["status"] == "FLAG":
            severity = result.get("severity", "WARNING")
            tag = f"[{severity}] {check_name}: {result['reason']}"
            if severity == "CRITICAL":
                critical_reasons.append(tag)
            elif severity == "WARNING":
                warning_reasons.append(tag)
            else:
                info_notes.append(tag)

    if readability.get("fkgl") is not None and readability["fkgl"] > MAX_FKGL_GRADE:
        warning_reasons.append(
            f"[WARNING] readability: FKGL grade {readability['fkgl']} exceeds {MAX_FKGL_GRADE}")
    if readability.get("fre") is not None and readability["fre"] < FRE_MINIMUM:
        warning_reasons.append(
            f"[WARNING] readability: FRE {readability['fre']} below {FRE_MINIMUM}")

    if word_count < MIN_WORD_COUNT:
        warning_reasons.append(
            f"[WARNING] word_count: {word_count} below minimum {MIN_WORD_COUNT}")
    elif word_count > MAX_WORD_COUNT:
        warning_reasons.append(
            f"[WARNING] word_count: {word_count} exceeds maximum {MAX_WORD_COUNT}")

    all_reasons = critical_reasons + warning_reasons + info_notes

    if critical_reasons:
        return "CRITICAL", all_reasons
    elif len(warning_reasons) >= 2:
        return "ESCALATE", all_reasons
    elif len(warning_reasons) == 1:
        return "REVIEW", all_reasons
    else:
        return "APPROVED", info_notes

print("✓ Tiered human review ready.")

✓ Tiered human review ready.


## Step 7: Main Pipeline (4 Stages)
```
[1/4] Two-Shot Summarization   → ~2-5 min (Phi-3 Mini, FULL text input)
[2/4] Readability Scoring      → instant (textstat)
[3/4] 10 Heuristic Checks     → instant (Python)
[4/4] Tiered Human Review      → instant (Python)
```


In [8]:
def process_single_text(health_text):
    """Full HEAL-Summ-Lite pipeline — two-shot prompting, 4 stages, no trimming."""
    text_id = health_text["id"]
    source = health_text["source"]
    title = health_text["title"]
    original = health_text["text"]

    print(f"\n{'='*70}")
    print(f"  Processing: {text_id} — {title}")
    print(f"  Source: {source}")
    print(f"  Original: {len(original.split())} words")
    print(f"{'='*70}")

    # --- Stage 1: Summarization ---
    print("\n  [1/4] Two-shot summarization (Phi-3 Mini, temp=0.3)...")
    print("        This may take 2-5 minutes on CPU. Please wait...")
    start_time = time.time()
    summary_prompt = build_summarization_prompt(original)
    raw_summary = call_ollama(summary_prompt, temperature=SUMMARIZATION_TEMPERATURE)
    if not raw_summary:
        print("        ❌ Summary generation failed.")
        return None
    gen_time = round(time.time() - start_time, 1)
    summary = clean_incomplete_output(raw_summary)
    was_cleaned = (summary != raw_summary)
    word_count = len(summary.split())
    if was_cleaned:
        print(f"        ✓ Done in {gen_time}s — {word_count} words (incomplete fragment removed)")
    else:
        print(f"        ✓ Done in {gen_time}s — {word_count} words (complete, no cleanup needed)")

    # --- Stage 2: Readability ---
    print("  [2/4] Computing readability metrics...")
    readability = compute_readability(summary)
    print(f"        FKGL: {readability.get('fkgl')} (grade level) | "
          f"FRE: {readability.get('fre')} (reading ease)")

    # --- Stage 3: 10 Heuristic Checks ---
    print("  [3/4] Running 10 rule-based heuristic checks...")
    heuristics = run_all_heuristics(original, summary, source)
    flags = sum(1 for v in heuristics.values() if v["status"] == "FLAG")
    passes = sum(1 for v in heuristics.values() if v["status"] == "PASS")
    print(f"        Results: {passes} PASS, {flags} FLAG")
    for name, result in heuristics.items():
        marker = "✓" if result["status"] == "PASS" else "⚠"
        sev = result.get("severity", "")
        print(f"          {marker} [{sev:8s}] {name}: {result['reason']}")

    # --- Stage 4: Tiered Human Review ---
    print("  [4/4] Applying tiered human review rules...")
    decision, review_reasons = decide_human_review(
        word_count, readability, heuristics)
    print(f"        Decision: {decision}")
    if review_reasons:
        for reason in review_reasons:
            print(f"          ⚠ {reason}")

    return {
        "id": text_id, "source": source, "title": title,
        "original_word_count": len(original.split()),
        "summary": summary, "summary_word_count": word_count,
        "fkgl": readability.get("fkgl"), "fre": readability.get("fre"),
        "heuristics": heuristics, "decision": decision,
        "review_reasons": review_reasons, "generation_time": gen_time,
    }

print("✓ Pipeline function ready.")

✓ Pipeline function ready.


## Step 8: Display Results

In [9]:
def display_results_table(results):
    """Formats and prints results in clean tables."""

    print("\n\n" + "=" * 100)
    print("  HEAL-Summ-Lite — RESULTS")
    print("=" * 100)

    # --- Table 1: Summary Overview ---
    print("\n  [TABLE 1] Summary Overview & Readability")
    print("  " + "-" * 80)
    t1_data = []
    for r in results:
        t1_data.append([
            r["id"],
            r["title"][:25] + ".." if len(r["title"]) > 25 else r["title"],
            r["original_word_count"],
            r["summary_word_count"],
            r["fkgl"],
            r["fre"],
            f"{r['generation_time']}s",
            r["decision"],
        ])
    h1 = ["ID", "Title", "Orig\nWords", "Summary\nWords", "FKGL", "FRE", "Time", "Decision"]
    print(tabulate(t1_data, headers=h1, tablefmt="grid", stralign="center"))

    # --- Table 2: Heuristic Results ---
    print("\n  [TABLE 2] Heuristic Quality Checks (10 Checks)")
    print("  " + "-" * 80)
    t2_data = []
    check_order = [
        "numeric_coverage", "entity_coverage", "compression_ratio",
        "caveat_presence", "negation_flip", "hedging_preservation",
        "source_attribution", "sentence_count", "semantic_similarity",
        "sensitive_topics",
    ]
    for r in results:
        h = r["heuristics"]
        row = [r["id"]]
        for check_name in check_order:
            row.append(h[check_name]["status"])
        t2_data.append(row)
    h2 = ["ID", "Numeric\nCover.", "Entity\nCover.", "Compress.\nRatio", "Caveat",
          "Negation\nFlip", "Hedging\nPreserv.", "Source\nAttrib.", "Sentence\nCount",
          "Semantic\nSimilar.", "Sensitive\nTopics"]
    print(tabulate(t2_data, headers=h2, tablefmt="grid", stralign="center"))

    # --- Flagged summaries ---
    flagged = [r for r in results if r["decision"] in ("CRITICAL", "ESCALATE")]
    if flagged:
        print(f"\n\n  FLAGGED FOR HUMAN REVIEW: {len(flagged)} of {len(results)} summaries")
        print("  " + "-" * 70)
        for r in flagged:
            print(f"\n    [{r['decision']}] {r['id']} — {r['title']}")
            for reason in r["review_reasons"]:
                print(f"      ⚠ {reason}")
    else:
        print("\n  ✓ All summaries passed quality checks.")

    # --- Print summaries ---
    print("\n\n" + "=" * 100)
    print("  GENERATED SUMMARIES")
    print("=" * 100)
    for r in results:
        print(f"\n  --- {r['id']}: {r['title']} ---")
        print(f"  [{r['summary_word_count']} words | FKGL: {r['fkgl']} | "
              f"FRE: {r['fre']} | {r['decision']}]")
        print(f"\n  {r['summary']}")

print("Display functions ready.")

Display functions ready.


## 🚀 Step 9: Select a Text and Run!

**Run the cell below.** Type a number (1-5) to summarize one text, or `A` for all.

In [10]:
if not HEALTH_TEXTS:
    print("No health texts loaded.")
    raise SystemExit

print("=" * 60)
print("  HEAL-Summ-Lite: Select a Health Text")
print("=" * 60)
print()

for i, ht in enumerate(HEALTH_TEXTS):
    wc = len(ht["text"].split())
    print(f"  [{i+1}] {ht['id']} — {ht['title']}")
    print(f"      Source: {ht['source']} | Words: {wc}")
    print()

print(f"  [A] Run ALL {len(HEALTH_TEXTS)} texts (~15-25 min)")
print()

choice = input("  Select (1-5, A=all): ").strip()

if choice.upper() == "A":
    selected = HEALTH_TEXTS
    print(f"\n  Running all {len(HEALTH_TEXTS)} texts...")
else:
    try:
        idx = int(choice) - 1
        if 0 <= idx < len(HEALTH_TEXTS):
            selected = [HEALTH_TEXTS[idx]]
            print(f"\n  Selected: {selected[0]['id']} — {selected[0]['title']}")
        else:
            selected = []
            print("  Invalid choice.")
    except ValueError:
        selected = []
        print("  Invalid choice.")

if not selected:
    print("\n No valid texts selected.")
else:
    results = []
    for ht in selected:
        result = process_single_text(ht)
        if result:
            results.append(result)

    if results:
        display_results_table(results)
        with open("heal_summ_results.json", "w") as f:
            json.dump(results, f, indent=2, default=str)
        print("\n  ✓ Results saved to heal_summ_results.json")

  HEAL-Summ-Lite: Select a Health Text

  [1] TEXT_01 — Diabetes – Key Facts and Prevention
      Source: World Health Organization (WHO) | Words: 605

  [2] TEXT_02 — About Influenza (Flu) – Symptoms, Spread, and Prevention
      Source: Centers for Disease Control and Prevention (CDC) | Words: 607

  [3] TEXT_03 — High Blood Pressure (Hypertension) – Causes, Risks and Treatment
      Source: National Health Service (NHS), UK | Words: 638

  [4] TEXT_04 — Malaria – Key Facts, Prevention, and Treatment
      Source: World Health Organization (WHO) | Words: 657

  [5] TEXT_05 — Mental Health – Understanding and Coping with Stress
      Source: Centers for Disease Control and Prevention (CDC) | Words: 693

  [A] Run ALL 5 texts (~15-25 min)



  Select (1-5, A=all):  4



  Selected: TEXT_04 — Malaria – Key Facts, Prevention, and Treatment

  Processing: TEXT_04 — Malaria – Key Facts, Prevention, and Treatment
  Source: World Health Organization (WHO)
  Original: 657 words

  [1/4] Two-shot summarization (Phi-3 Mini, temp=0.3)...
        This may take 2-5 minutes on CPU. Please wait...
        ✓ Done in 330.0s — 176 words (incomplete fragment removed)
  [2/4] Computing readability metrics...
        FKGL: 16.2 (grade level) | FRE: 30.9 (reading ease)
  [3/4] Running 10 rule-based heuristic checks...
        Results: 4 PASS, 6 FLAG
          ⚠ [WARNING ] numeric_coverage: Only 21.7% of numbers preserved (18 missing)
          ⚠ [WARNING ] entity_coverage: Only 30.8% of entities preserved (9 missing)
          ✓ [INFO    ] compression_ratio: Compression ratio 0.27 — within acceptable range
          ⚠ [CRITICAL] caveat_presence: Treatment content found but NO safety caveat in summary
          ✓ [INFO    ] negation_flip: No negation flips detected
      

In [11]:
# Run ALL 5 texts through the pipeline
results = []
total_start = time.time()

for i, health_text in enumerate(HEALTH_TEXTS):
    print(f"\n{'#'*70}")
    print(f"  TEXT {i+1} of {len(HEALTH_TEXTS)}")
    print(f"{'#'*70}")
    result = process_single_text(health_text)
    if result:
        results.append(result)

total_time = round(time.time() - total_start, 1)
print(f"\n\n{'='*70}")
print(f"  ALL DONE — {len(results)} texts processed in {total_time}s")
print(f"{'='*70}")

# Display results
if results:
    display_results_table(results)
    with open("heal_summ_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    print("\n  Results saved to heal_summ_results.json")


######################################################################
  TEXT 1 of 5
######################################################################

  Processing: TEXT_01 — Diabetes – Key Facts and Prevention
  Source: World Health Organization (WHO)
  Original: 605 words

  [1/4] Two-shot summarization (Phi-3 Mini, temp=0.3)...
        This may take 2-5 minutes on CPU. Please wait...
        ✓ Done in 319.8s — 209 words (incomplete fragment removed)
  [2/4] Computing readability metrics...
        FKGL: 17.3 (grade level) | FRE: 33.0 (reading ease)
  [3/4] Running 10 rule-based heuristic checks...
        Results: 7 PASS, 3 FLAG
          ⚠ [WARNING ] numeric_coverage: Only 21.1% of numbers preserved (15 missing)
          ✓ [INFO    ] entity_coverage: 83.3% of entities preserved
          ✓ [INFO    ] compression_ratio: Compression ratio 0.35 — within acceptable range
          ⚠ [CRITICAL] caveat_presence: Treatment content found but NO safety caveat in summary
          ✓ 